# Agenci LLM w tworzeniu kodu

### Wprowadzenie, metodyka pracy i praktyczne wnioski

---

**EW — Energetyka Wodna, 2025/2026**

*Ten notebook jest prezentacją — użyj RISE lub `View → Slideshow` w VS Code*

## Konfiguracja środowiska pracy

Zanim zaczniemy pracę z kodem i AI, potrzebujemy skonfigurować środowisko.

### Python — środowiska wirtualne

**Problem:** Różne projekty wymagają różnych wersji bibliotek. Instalowanie wszystkiego globalnie prowadzi do konfliktów.

**Rozwiązanie:** Środowiska wirtualne — izolowane "kopie" Pythona per projekt.

| Narzędzie | Opis | Kiedy użyć |
|-----------|------|------------|
| **venv** | Wbudowane w Python 3 | Proste projekty, tylko pakiety pip |
| **conda/miniconda** | Zarządza Pythonem + bibliotekami C/Fortran | Nauka danych, NumPy, SciPy |
| **poetry** | Nowoczesny menedżer zależności | Duże projekty, publikowanie pakietów |

```bash
# Miniconda (używamy w tym projekcie)
conda create -n ew python=3.11 pandas numpy plotly jupyter
conda activate ew

# Alternatywnie: venv + pip
python -m venv venv
source venv/bin/activate      # Linux/Mac
venv/Scripts/activate          # Windows
pip install -r requirements.txt
```

**Ważne:** Środowisko to NIE jest częścią repozytorium. Plik `requirements.txt` lub `environment.yml` opisuje co zainstalować.

### VS Code — edytor i rozszerzenia

**Visual Studio Code** to darmowy edytor od Microsoft, idealny do pracy z Pythonem i Jupyter.

Kluczowe rozszerzenia:

| Rozszerzenie | Do czego |
|-------------|----------|
| **Python** | Wsparcie języka, debugger, linter |
| **Jupyter** | Edycja i uruchamianie notebooków |
| **Pylance** | Inteligentne podpowiedzi kodu |
| **GitLens** | Rozszerzona integracja z Git |
| **GitHub Copilot** | Agent AI do kodu (opcjonalnie) |
| **RISE** | Prezentacje z notebooków Jupyter (pip install rise) |

**Konfiguracja kernela:** Po otwarciu notebooka → prawy górny róg → `Select Kernel` → wybierz środowisko `ew` (conda).

### Agenci AI do kodu — alternatywy

Oprócz Claude Code istnieje kilka konkurencyjnych narzędzi. Oto porównanie:

| Narzędzie | Dostawca | Model | Koszt |
|-----------|----------|-------|-------|
| **Claude Code** | Anthropic | Claude 4 | Subskrypcja Pro/Max |
| **Codex CLI** | OpenAI | GPT-4o / o3 | API key (pay-per-use) |
| **Gemini CLI** | Google | Gemini 2.5 | Darmowy (z limitem) |
| **GitHub Copilot** | Microsoft/OpenAI | GPT-4o | Darmowy dla studentów! |
| **Cursor** | Cursor Inc. | Wiele modeli | Darmowy tier + Pro |

**Instalacja agentów CLI** (wymagany Node.js + npm):
```bash
# Claude Code
npm install -g @anthropic-ai/claude-code

# OpenAI Codex CLI
npm install -g @openai/codex

# Gemini CLI
npm install -g @google/gemini-cli
# Klucz API: https://aistudio.google.com
```

**GitHub Copilot (zalecane dla studentów):**
1. Zarejestruj się na github.com/education (darmowe z mailem uczelnianym)
2. VS Code → Extensions → zainstaluj "GitHub Copilot"
3. Zaloguj się kontem GitHub → gotowe

**Google Gemini na uczelni (PWr):**
Gemini jest dostępny w ramach **Google Workspace for Education Plus**.
Wystarczy zalogować się kontem uczelnianym Google i zaakceptować regulamin.
Dane z rozmów **nie są** wykorzystywane do trenowania modelu.
Nie udostępniaj informacji poufnych! Szczegóły: di.pwr.edu.pl/oprogramowanie/ai/google-gemini

> Dla studentów najłatwiej zacząć z **GitHub Copilot** (darmowy) lub **Gemini** (przez konto uczelniane).

### Instalacja Node.js / npm

Agenci CLI (Claude Code, Codex, Gemini CLI) wymagają **Node.js** i **npm** (menedżer pakietów Node).

**Instalacja Node.js:**

| System | Instrukcja |
|--------|-----------|
| **Windows** | Pobierz instalator z https://nodejs.org (wersja LTS) |
| **macOS** | `brew install node` lub instalator z nodejs.org |
| **Linux (Ubuntu)** | `sudo apt install nodejs npm` |

Po instalacji sprawdź:
```bash
node --version    # np. v20.11.0
npm --version     # np. 10.2.4
```

**Instalacja agenta (np. Claude Code):**
```bash
npm install -g @anthropic-ai/claude-code
claude    # uruchomienie
```

**npm** instaluje pakiety JavaScript globalnie (`-g`) lub lokalnie w projekcie.
Agenci AI do kodu to aplikacje Node.js — stąd potrzeba npm.

### Git — system kontroli wersji

**Git** śledzi każdą zmianę w plikach projektu. Pozwala:
- Cofnąć się do dowolnej poprzedniej wersji
- Pracować równolegle nad różnymi funkcjami (gałęzie/branches)
- Współpracować zespołowo (GitHub, GitLab)

Podstawowe komendy:
```bash
git init                    # Inicjalizacja repozytorium
git add plik.py             # Dodaj plik do kolejnego commita
git commit -m "opis zmian"  # Zapisz zmiany z opisem
git status                  # Co się zmieniło?
git log --oneline           # Historia zmian
git diff                    # Pokaż różnice
```

**Ważne pliki:**
- `.gitignore` — lista plików/folderów do ignorowania (dane, venv, cache)
- `requirements.txt` / `environment.yml` — zależności projektu (wersjonowane!)

### Format danych: dlaczego Parquet?

W tym projekcie używamy formatu **Apache Parquet** zamiast CSV do przechowywania przetworzonych danych.

| Cecha | CSV | Parquet |
|-------|-----|--------|
| Format | Tekstowy | Binarny, kolumnowy |
| Rozmiar | Duży | 5-10x mniejszy (kompresja) |
| Typy danych | Brak (wszystko to tekst) | Zachowane (int, float, datetime) |
| Szybkość odczytu | Wolny (parsowanie tekstu) | Szybki (natywne typy) |
| Kodowanie | Problemy (CP-1250, UTF-8...) | Brak problemów (Unicode) |
| Obsługa w pandas | `read_csv()` | `read_parquet()` |

```python
# Zapis
df.to_parquet('dane.parquet')

# Odczyt — typy zachowane, nie trzeba parsować dat
df = pd.read_parquet('dane.parquet')
```

**Dlaczego wybraliśmy Parquet:**
- Dane IMGW mają problemy z kodowaniem (CP-1250 vs UTF-8) — Parquet to eliminuje
- Daty i typy numeryczne są zachowane — nie trzeba konwertować po każdym odczycie
- Pliki są mniejsze — ważne przy wieloletnich danych dobowych

### Struktura tego projektu

```
EW/
├── ai/                  # Pliki asystenta AI (CLAUDE.md, plan)
├── data/
│   ├── raw/             # Pobrane ZIP z IMGW (gitignored)
│   └── processed/       # Przetworzone Parquet (gitignored)
├── src/                 # Moduły Pythona wielokrotnego użytku
│   ├── imgw.py          # Pobieranie i parsowanie danych IMGW
│   ├── data_quality.py  # Detekcja luk, uzupełnianie, walidacja
│   └── hydro_stats.py   # Krzywe FDC, przepływy charakterystyczne
├── notebooks/           # Notebooki Jupyter (numerowane krokami)
├── environment.yml      # Definicja środowiska conda
├── requirements.txt     # Alternatywa pip
└── .gitignore           # Co git ma ignorować
```

**Zasada:** Kod wielokrotnego użytku → `src/`, analiza krok po kroku → `notebooks/`

## Czym jest agent kodujący LLM?

**Agent LLM** to system AI, który potrafi:

- Rozumieć instrukcje w języku naturalnym
- **Czytać** i **pisać** pliki z kodem
- **Uruchamiać** komendy w terminalu i analizować wyniki
- **Przeszukiwać** bazę kodu (grep, glob, web)
- **Iterować** — naprawiać błędy, ponownie uruchamiać, adaptować się

W odróżnieniu od zwykłego chatbota, agent **działa autonomicznie** w pętli:

```
Zapytanie → Myślenie → Użycie narzędzia → Obserwacja wyniku → Myślenie → ... → Gotowe
```

### Przykłady agentów kodujących (2025–2026)

| Narzędzie | Typ | Sposób działania |
|-----------|-----|------------------|
| **Claude Code** | Agent CLI | Działa w terminalu, czyta/pisze pliki, wykonuje komendy |
| **GitHub Copilot** | Integracja z IDE | Autouzupełnianie + czat w VS Code |
| **Cursor** | IDE z AI | Fork VS Code z głęboką integracją AI |
| **Aider** | Agent CLI | Open-source, świadomy gita, programowanie w parze |
| **Devin** | Agent autonomiczny | Próbuje realizować pełne zadania samodzielnie |
| **ChatGPT / Claude.ai** | Czat | Praca przez kopiuj-wklej, bez dostępu do plików |

## Jak powstał ten projekt

Cały zestaw narzędzi do analizy MEW został opracowany **wspólnie** z Claude Code:

```
Człowiek:  "Stwórz notebook do obliczeń hydrologicznych —
            pobierz dane IMGW, sprawdź jakość, oblicz statystyki..."

Agent:     1. Zbadał strukturę FTP/API IMGW (wyszukiwanie w sieci)
           2. Stworzył strukturę projektu (katalogi, git, conda env)
           3. Napisał moduły wielokrotnego użytku (src/imgw.py, ...)
           4. Zbudował notebooki z opisami + kodem + wykresami
           5. Obsłużył obie ery formatu CSV (przed 2023 vs 2023+)
```

Czas: ~2 godziny na działający projekt z wieloma iteracjami i naprawkami (vs dni ręcznie)

### Co agent zrobił dobrze

- Odkrył format danych IMGW **przeszukując internet** w czasie rzeczywistym
- Obsłużył problemy z kodowaniem (CP-1250 vs UTF-8), które człowiekowi zabrałyby czas
- Stworzył **modularną strukturę** (`src/` + `notebooks/`) z ogólnego opisu
- Napisał kod, docstringi i opisy w notebooku **jednocześnie**
- Skonfigurował infrastrukturę (git, conda, .gitignore) niejako "przy okazji"

## Zalety

### 1. Ogromny wzrost produktywności

- **Szkieletowanie** — konfiguracja projektu, boilerplate, pliki konfiguracyjne w sekundy
- **Integracja z API** — agent czyta dokumentację i pisze działający kod klienta
- **Obróbka danych** — pipeline'y pandas, konwersja formatów, czyszczenie
- **Wykresy** — generuje kod plotly/matplotlib z opisu słownego

> Zadanie trwające 2–4 godziny ręcznie można zrealizować w 15–30 minut.

Najlepiej sprawdza się przy: **dobrze zdefiniowanych zadaniach** z jasnymi danymi wejściowymi i wyjściowymi.

### 2. Akcelerator nauki

- Zapytaj *"dlaczego użyłeś transformacji logarytmicznej?"* — dostaniesz wyjaśnienie
- Poznawaj nowe biblioteki obserwując jak agent ich używa
- Ucz się idiomatycznych wzorców (np. pandas groupby, plotly subplots)
- **Ucz się przez przegląd kodu**, nie przez kopiowanie ze Stack Overflow

> Agent to niestrudzony partner do programowania w parze, który wszystko wyjaśni.

Uwaga: musisz **rozumieć** to, co produkuje — ślepe zaufanie mija się z celem.

### 3. Eksploracja i prototypowanie

- *"Wypróbuj trzy metody interpolacji i porównaj"* → zrobione w jednym prompcie
- Szybka iteracja: opisz → wygeneruj → testuj → udoskonal
- Świetne do **studiów wykonalności** — czy te dane są użyteczne? Czy to API działa?
- Agent może prowadzić **research podczas kodowania** (wyszukiwanie + kod w jednym)

### 4. Most między domeną a kodem

- Znasz **hydrologię** ale nie `pandas.groupby` → agent łączy te światy
- Znasz **wzór** (P = ρgQH) → agent zajmie się implementacją
- Zmniejsza frustrację *"wiem co chcę, ale nie umiem tego zakodować"*
- Szczególnie wartościowe dla **ekspertów dziedzinowych, nie będących programistami**

## Wady

### 1. Halucynacje i pewne siebie błędy

Agent potrafi:
- Wymyślić **endpointy API, które nie istnieją**
- Użyć **błędnych sygnatur funkcji** ze starych danych treningowych
- Wygenerować kod, który **wygląda poprawnie, ale ma subtelne błędy**
- Twierdzić, że biblioteka ma funkcję, której nie ma

**Prawdziwy przykład:** LLM może wygenerować `pd.read_csv(..., encoding='utf-8')` dla danych IMGW —
zadziała dla plików 2023+, ale **po cichu uszkodzi** polskie znaki w plikach sprzed 2023 (CP-1250).

> Im bardziej pewnie brzmi agent, tym uważniej powinieneś weryfikować.

### 2. Utrata zrozumienia

**Największe ryzyko w edukacji:**

- Student dostaje działający kod, nie rozumiejąc *dlaczego* działa
- Nie potrafi debugować, gdy coś się zepsuje w produkcji/terenie
- Pomija krzywą uczenia, która buduje **intuicję**
- *"Działa"* nie znaczy *"Rozumiem to"*

**Przykład:** Agent pisze funkcję krzywej uporządkowanej przepływu.
Jeśli nie rozumiesz, że `np.sort(values)[::-1]` daje kolejność malejącą,
nie zmodyfikujesz tego dla analizy okresu częściowego.

> Używaj agenta jako **tutora**, nie jako **zamiennika** dla zrozumienia.

### 3. Nadmierna inżynieria i nadmiar kodu

Agenci mają tendencję do:
- Dodawania **zbędnej obsługi błędów** dla przypadków, które nie mogą wystąpić
- Tworzenia **abstrakcji** dla operacji jednorazowych
- Dodawania **type hintów, docstringów, logowania** wszędzie (nawet gdy niepotrzebne)
- Generowania 200 linii, gdy wystarczyłoby 20

**Efekt:** Kod trudniejszy do czytania i utrzymania niż pisany ręcznie.

> Zawsze przeglądaj i upraszczaj. Najlepszy kod to najmniej kodu, który działa.

### 4. Bezpieczeństwo i prywatność danych

- Kod wysyłany do API w chmurze → **nie wklejaj poświadczeń ani wrażliwych danych**
- Agent może wygenerować kod z lukami: **SQL injection, XSS, command injection**
- Może sugerować `pip install` pakietów, które są **złośliwe** (typosquatting)
- Pobrany kod nie jest audytowany — **odpowiedzialność ponosi użytkownik**

> Traktuj kod generowany przez AI z taką samą ostrożnością jak kod od nieznanego autora.

### 5. Problemy z odtwarzalnością

- Ten sam prompt → **inny kod** za każdym razem (niedeterministyczne)
- Wiedza agenta ma **datę odcięcia** — może nie znać ostatnich zmian API
- Brak formalnej **proweniencji** — kto to napisał? Człowiek czy AI? Która wersja?
- Kod naukowy wymaga **odtwarzalności** — asystent AI to komplikuje

> Zawsze używaj systemu kontroli wersji. Dokumentuj, które części napisało AI.

## Częste problemy i pułapki

### Problem: "Działa na moim komputerze"

Agent generuje kod, który działa w **jego kontekście**, ale nie gdzie indziej:

- Ścieżki na sztywno (np. `/home/jan/data/` lub `C:/Users/jan/data/`)
- Zakłada zainstalowane pakiety, których nie ma w `requirements.txt`
- Testowane z jednym kształtem danych, psuje się na innym
- Kod specyficzny dla platformy (separatory ścieżek Windows vs Linux)

**Rozwiązanie:** Zawsze uruchamiaj kod samodzielnie. Testuj z różnymi danymi wejściowymi.

### Problem: "Nieskończona pętla poprawek"

```
Agent pisze kod → Błąd → Poprawia → Nowy błąd → Poprawia → ...
```

Czasem agent:
- Nie rozumie **pierwotnej przyczyny**
- Nakłada łatki, które wprowadzają nowe błędy
- Krąży próbując różnych podejść bez cofnięcia się

**Rozwiązanie:** Jeśli agent utknął po 2–3 próbach, **zatrzymaj się i pomyśl sam.**
Opisz faktyczny problem precyzyjniej lub rozbij na mniejsze kroki.

### Problem: "Nieaktualna wiedza"

LLM mają **datę odcięcia danych treningowych**. Mogą:

- Używać przestarzałych API (`pandas.append()` usunięte w 2.0)
- Nie znać najnowszych wersji bibliotek
- Odwoływać się do dokumentacji, która się zmieniła
- Nie wiedzieć o zmianach formatu (jak przejście IMGW z przecinka na średnik w 2023)

**Rozwiązanie:** Agenci z **wyszukiwaniem w sieci** mogą to kompensować —
ale zawsze weryfikuj z aktualną dokumentacją.

### Problem: "Luka w ekspertyzie dziedzinowej"

Agent dobrze zna Pythona, ale może nie wiedzieć, że:

- **Rok hydrologiczny** zaczyna się w listopadzie, nie w styczniu
- Przepływ = 0 może być **poprawny** (cieki okresowe) lub **błędem**
- 99.9, 9999, 99999 to **wartości strażnikowe IMGW** oznaczające brak danych
- Q90 oznacza przepływ przekraczany przez **90%** czasu (nie 90. percentyl)

**Rozwiązanie:** To ty jesteś **ekspertem dziedzinowym**. Agent jest asystentem programistycznym.
Zawsze weryfikuj logikę specyficzną dla domeny samodzielnie.

### Prawdziwe problemy z tego projektu

Podczas budowy tego projektu agent popełnił błędy, ktore musieliśmy naprawić:

| Problem | Co agent zrobił źle | Naprawa |
|---------|---------------------|---------|
| **Kodowanie CSV** | Założył UTF-8 dla plików 2023+ | Niektóre nadal CP-1250 - dodano auto-detekcję |
| **Cudzysłowy CSV** | Ustawił QUOTE_NONE dla wszystkich | Stare pliki mają cudzysłowy - ID czytane z nimi |
| **Wartosci strażnikowe** | Nie wiedział o 9999/99999 IMGW | Przepływ 99999 = brak danych, nie rekord! |
| **JSON w notebookach** | Backslash w ścieżkach Windows | Psuje JSON — trzeba generować przez Python |
| **Gemini CLI** | Wkleił pakiet Claude zamiast Google | Kopiuj-wklej błąd - trzeba weryfikować! |
| **Moduły w src/** | Pisał cały kod w notebooku | Trzeba bylo przypominać o modularności |

> Te błędy to **normalna część** pracy z agentem. Kluczowa jest **weryfikacja** i iteracja.

## Dobre praktyki

### Efektywne używanie agentów LLM

1. **Bądź konkretny** — *"pobierz dobowe dane przepływu z IMGW dla stacji 149180080, lata 1990–2024"*
   jest lepsze niż *"pobierz jakieś dane hydro"*

2. **Iteruj małymi krokami** — nie proś o wszystko na raz. Buduj przyrostowo.

3. **Przeglądaj każdą linię** — szczególnie:
   - Logikę parsowania danych (kodowania, mapowania kolumn)
   - Wzory matematyczne
   - Przypadki brzegowe (puste dane, braki, dzielenie przez zero)

4. **Testuj ze znanymi danymi** — weryfikuj obliczenia z przykładami z podręcznika

5. **Wersjonuj wszystko** — commituj często, żebyś mógł cofnąć błędy AI

### Dla studentów

1. **Czytaj zanim uruchomisz** — zrozum kod przed jego wykonaniem

2. **Pytaj "dlaczego"** — każ agentowi wyjaśnić swoje wybory

3. **Modyfikuj i psuj** — zmień parametry, usuń linie, zobacz co się stanie

4. **Najpierw zrób sam** — spróbuj napisać funkcję, potem porównaj z wersją AI

5. **Kwestionuj agenta** — *"Czy to najbardziej efektywny sposób?"*, *"Co jeśli dane mają luki?"*

> Celem jest **wzmocnienie** twoich umiejętności, nie ich zastąpienie.

### Dla prowadzących

1. **Zaakceptuj to** — studenci będą używać AI niezależnie; naucz ich używać dobrze

2. **Wymagaj wyjaśnienia** — *"Oddaj kod I wyjaśnij jak działa"*

3. **Różnicuj zadania** — identyczne prompty → identyczny kod → łatwe do wykrycia

4. **Testuj zrozumienie** — egzaminy ustne, live coding, ćwiczenia "zmodyfikuj i wyjaśnij"

5. **Używaj AI na zajęciach** — demonstracje na żywo, pokazuj porażki i ograniczenia

6. **Dyskutuj o etyce** — autorstwo, atrybucja, uczciwość akademicka

## Dokąd to zmierza?

### Teraz (2025–2026)
- Agenci potrafią tworzyć szkielety projektów, pisać moduły, naprawiać błędy
- Wciąż wymagają **nadzoru człowieka** dla poprawności
- Najlepsi jako **programista w parze**, nie jako autonomiczny deweloper

### Bliska przyszłość (2026–2028)
- Lepsza wiedza dziedzinowa (modele dostrojone dla inżynierii)
- Agenci, którzy **uruchamiają i testują** swój kod przed dostarczeniem
- Integracja z narzędziami symulacyjnymi (HEC-RAS, MIKE, QGIS)

### Stała rzecz
- **Ekspertyza dziedzinowa pozostaje niezastąpiona**
- Inżynier, który rozumie hydrologię I potrafi efektywnie kierować AI,
  zawsze będzie lepszy niż każde z osobna

## Praktyczna demonstracja: co agent generuje

Poniżej przykłady kodu wygenerowanego przez agenta LLM.
Przed każdym blokiem kodu — **prompt**, który mógłby go wygenerować.

Zwróć uwagę: ten sam prompt może dać różny kod przy każdym uruchomieniu!

### Przykład 1: Obliczanie mocy MEW

**Prompt do LLM:**
> *"Napisz funkcję w Pythonie obliczającą moc elektrowni wodnej ze wzoru P = ρgQHη.
> Parametry: przepływ Q, spad netto H, sprawność eta. Wynik w kW. Dodaj przykład użycia."*

In [ ]:
# Kod wygenerowany przez LLM na podstawie powyzszego promptu
import numpy as np

def calculate_power(Q, H, eta=0.85, rho=1000, g=9.81):
    """
    Oblicz moc elektrowni wodnej.
    P = rho * g * Q * H * eta

    Args:
        Q: przeplyw [m3/s]
        H: spad netto [m]
        eta: sprawnosc calkowita [-]
        rho: gestosc wody [kg/m3]
        g: przyspieszenie ziemskie [m/s2]
    Returns:
        Moc [kW]
    """
    return rho * g * Q * H * eta / 1000

# Przyklad
Q = 5.0   # m3/s
H = 10.0  # m
P = calculate_power(Q, H)
print(f'Q = {Q} m3/s, H = {H} m, P = {P:.1f} kW')

### Na co zwrócić uwagę w tym kodzie

- Wzór jest poprawny: P = ρgQHη / 1000 → kW
- Brak walidacji danych wejściowych: ujemne Q? H = 0?
- `eta = 0.85` — czy to rozsądna wartość domyślna? (turbina? generator? transformator?)
- `rho = 1000` — zakłada czystą wodę słodką przy ~4 C
- Brak obsługi tablic — co jeśli Q jest pd.Series?

**Kluczowy wniosek:** Kod *działa* i jest *w większości poprawny*, ale wartości domyślne
kodują **założenia**, które mogą nie pasować do twojego przypadku.

### Przykład 2: Interaktywna mapa stacji

**Prompt do LLM:**
> *"Mam DataFrame z kolumnami lat, lon, stacja, rzeka. Stwórz interaktywną mapę
> w plotly z tymi punktami na tle OpenStreetMap. Koloruj po rzece, dodaj hover z nazwą stacji."*

In [ ]:
# Kod wygenerowany przez LLM na podstawie powyzszego promptu
import pandas as pd
import plotly.express as px

# Przykladowe dane (w prawdziwym projekcie z API IMGW)
demo_data = pd.DataFrame({
    'lat': [49.62, 49.85, 50.06, 50.26, 49.98],
    'lon': [18.92, 19.49, 19.94, 19.79, 20.06],
    'stacja': ['Wisla Czarne', 'Zywiec', 'Krakow', 'Nowy Bierun', 'Proszowki'],
    'rzeka': ['Wisla', 'Sola', 'Wisla', 'Wisla', 'Raba'],
})

fig = px.scatter_mapbox(
    demo_data,
    lat='lat', lon='lon',
    hover_name='stacja',
    hover_data=['rzeka'],
    color='rzeka',
    zoom=7, height=500,
    title='Przykladowa mapa stacji hydrologicznych'
)
fig.update_layout(mapbox_style='open-street-map')
fig.show()

### Przykład 3: Krzywa uporządkowana przepływu

**Prompt do LLM:**
> *"Napisz funkcję, która z szeregu dobowych przepływów oblicza krzywą uporządkowaną przepływu (FDC).
> Zwróć DataFrame z kolumnami: przepływ i procent przekroczenia.
> Pokaż wynik na wykresie z osią Y logarytmiczną."*

In [ ]:
# Kod wygenerowany przez LLM na podstawie powyzszego promptu
import numpy as np
import pandas as pd
import plotly.graph_objects as go

def flow_duration_curve_demo(discharge_values):
    """Oblicz krzywa uporzadkowana przeplywu."""
    sorted_q = np.sort(discharge_values)[::-1]
    n = len(sorted_q)
    exceedance = np.arange(1, n + 1) / n * 100
    return pd.DataFrame({'discharge_m3s': sorted_q, 'exceedance_pct': exceedance})

# Wygeneruj przykladowe dane (rozklad log-normalny, typowy dla przeplywow)
np.random.seed(42)
sample_q = np.random.lognormal(mean=2.0, sigma=0.8, size=3650)  # ~10 lat

fdc = flow_duration_curve_demo(sample_q)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fdc['exceedance_pct'], y=fdc['discharge_m3s'],
    mode='lines', line=dict(color='royalblue', width=2),
))
fig.update_layout(
    title='Krzywa uporzadkowana przeplywu (dane przykladowe)',
    xaxis_title='Prawdopodobienstwo przekroczenia [%]',
    yaxis_title='Q [m3/s]', yaxis_type='log', height=450,
)
fig.show()

## Podsumowanie

| Aspekt | Agenci LLM |
|--------|------------|
| **Najlepsi w** | Szkieletowanie, boilerplate, obróbka danych, eksploracja |
| **Dobrzy w** | Wyjaśnianie kodu, łączenie domeny z kodem, szybkie prototypy |
| **Ryzykowni w** | Logika dziedzinowa, przypadki brzegowe, bezpieczeństwo |
| **Słabi w** | Zastępowanie zrozumienia, gwarantowana poprawność, nowe algorytmy |

### Wniosek

> **Agent LLM to potężne narzędzie — ale narzędzie jest tylko tak dobre, jak osoba, która go używa.**
>
> Naucz się podstaw. Używaj AI żeby być szybszym. Zawsze weryfikuj.

---

*Ta prezentacja została wygenerowana z pomocą Claude Code — a następnie zweryfikowana przez człowieka.*